# 🦴 Bone Fracture 10-Class Classification
## Transfer Learning with EfficientNet-B3

---

## 📚 전체 학습 흐름

| 단계 | 셀 번호 | 내용 |
|------|---------|------|
| **환경 설정** | CELL 1~2 | 라이브러리 설치, Google Drive 마운트, 경로 설정 |
| **데이터 탐색** | CELL 3~5 | 폴더 구조 분석, 샘플 시각화, 클래스 분포 |
| **전처리** | CELL 6~8 | 하이퍼파라미터, 데이터 증강, 데이터로더 |
| **모델** | CELL 9~11 | 모델 정의, 손실함수, 옵티마이저 |
| **학습** | CELL 12~14 | 학습 루프, Early Stopping, 체크포인트 저장 |
| **평가** | CELL 15~17 | 테스트, Confusion Matrix, Classification Report |
| **시각화** | CELL 18~19 | Per-Class 정확도, 학습 곡선 |
| **결과 저장** | CELL 20 | 요약 JSON 저장, 최종 결과 출력 |

---
> ✅ **GPU 사용 권장**: 런타임 → 런타임 유형 변경 → T4 GPU 선택

---
## 🔧 STEP 1. 환경 설정 (Environment Setup)

### CELL 1-A: 라이브러리 설치

**학습 포인트:**
- `torchvision` : PyTorch 공식 이미지 처리 라이브러리 (데이터셋, 변환, 사전학습 모델)
- `timm` : Hugging Face의 PyTorch Image Models — 최신 SOTA 모델 700+ 제공
- `scikit-learn` : 머신러닝 평가 지표 (confusion_matrix, classification_report 등)
- `tqdm` : 학습 진행 상황 표시 (progress bar)

> 💡 `!pip install -q` 에서 `-q`는 quiet 모드 (출력 최소화)

In [1]:
# ============================================================
# CELL 1-A: 필수 라이브러리 설치
# ============================================================
# Colab에는 기본적으로 PyTorch가 설치되어 있음
# 추가로 필요한 라이브러리만 설치

!pip install -q timm          # PyTorch Image Models (EfficientNet 등)
!pip install -q seaborn       # 시각화 라이브러리 (Confusion Matrix용)

print("✅ 라이브러리 설치 완료")

✅ 라이브러리 설치 완료


### CELL 1-B: Google Drive 마운트

**학습 포인트:**
- Colab은 세션이 끊기면 로컬 파일이 모두 삭제됨
- Google Drive를 마운트하면 `/content/drive/MyDrive/` 경로로 파일 영구 저장 가능
- 최초 실행 시 Google 계정 인증 팝업이 뜸

In [2]:
# ============================================================
# CELL 1-B: Google Drive 마운트
# ============================================================
from google.colab import drive

drive.mount('/content/drive')
print("✅ Google Drive 마운트 완료")
print("   📁 접근 경로: /content/drive/MyDrive/")

Mounted at /content/drive
✅ Google Drive 마운트 완료
   📁 접근 경로: /content/drive/MyDrive/


### CELL 2: 경로 설정 및 출력 폴더 생성

**학습 포인트:**
- `INPUT_PATH` : 학습 데이터가 있는 폴더 (각자 환경에 맞게 수정!)
- `OUTPUT_BASE` : 결과물을 저장할 기본 폴더
- `RUN_STAMP` : 실행할 때마다 타임스탬프 기반 하위 폴더 자동 생성 → 결과물 덮어쓰기 방지
- `os.makedirs(..., exist_ok=True)` : 폴더가 없으면 생성, 있으면 오류 없이 통과

> ⚠️ `INPUT_PATH`는 본인 Drive 경로로 반드시 수정하세요!

In [ ]:
# ============================================================
# CELL 2: 경로 설정 및 출력 폴더 생성
# ============================================================
import os
from datetime import datetime

# ── ⚠️ 본인 Drive 경로로 수정하세요 ──────────────────────────
INPUT_PATH  = "/content/drive/MyDrive/2026_lecture/Medical_AI/Medical_Imagining/Bone Break Classification"
OUTPUT_BASE = "/content/drive/MyDrive/2026_lecture/Medical_AI/1week/output"

# ── 타임스탬프 기반 출력 폴더 (실행마다 새 폴더 생성) ──────────
RUN_STAMP   = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_PATH = os.path.join(OUTPUT_BASE, f"run_{RUN_STAMP}")
os.makedirs(OUTPUT_PATH, exist_ok=True)

print(f"✅ 경로 설정 완료")
print(f"   📂 입력(데이터): {INPUT_PATH}")
print(f"   💾 출력(결과물): {OUTPUT_PATH}")

# ── 경로 유효성 확인 ──────────────────────────────────────────
if os.path.exists(INPUT_PATH):
    print(f"   ✅ 입력 경로 확인됨")
else:
    print(f"   ❌ 입력 경로가 존재하지 않습니다! 경로를 확인하세요.")

---
## 🔍 STEP 2. 데이터 탐색 (EDA: Exploratory Data Analysis)

### CELL 3: 클래스(폴더) 목록 및 이미지 수 파악

**학습 포인트:**
- ImageFolder 형식: 폴더명 = 클래스명 (표준 구조)
- `pathlib.Path.rglob("*")` : 하위 폴더까지 재귀적으로 모든 파일 탐색
- 클래스 불균형(Class Imbalance)이 있는지 확인 → 모델 편향 방지 전략 필요
- 지원 확장자: `.png`, `.jpg`, `.jpeg`, `.bmp`, `.tiff`, `.tif`, `.webp`

**데이터 폴더 구조 예시:**
```
Bone Break Classification/
├── Avulsion fracture/    ← 클래스 1
├── Comminuted fracture/  ← 클래스 2
├── ...
└── Stress fracture/      ← 클래스 10
```

In [ ]:
# ============================================================
# CELL 3: 클래스 목록 및 이미지 수 파악
# ============================================================
import pathlib
from PIL import Image

# 지원하는 이미지 확장자 목록
IMG_EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.tif', '.webp'}

print("=" * 60)
print("📂 폴더 구조 분석")
print("=" * 60)

# 서브폴더(클래스) 목록 추출 및 정렬
class_names = sorted([
    d for d in os.listdir(INPUT_PATH)
    if os.path.isdir(os.path.join(INPUT_PATH, d))
])

print(f"✅ 감지된 클래스 수: {len(class_names)}")
print()

# 클래스별 이미지 수 집계
total_images = 0
class_counts = {}      # {클래스명: 이미지 수}
sample_images = {}     # {클래스명: 대표 이미지 경로}

for cls in class_names:
    cls_dir = os.path.join(INPUT_PATH, cls)
    imgs = [
        f for f in pathlib.Path(cls_dir).rglob("*")
        if f.suffix.lower() in IMG_EXTS
    ]
    count = len(imgs)
    class_counts[cls] = count
    total_images += count
    if imgs:
        sample_images[cls] = str(imgs[0])  # 첫 번째 이미지를 샘플로
    print(f"  [{cls:35s}] {count:5d} 이미지")

print("-" * 60)
print(f"  {'TOTAL':35s} {total_images:5d} 이미지")
print()

# 클래스 불균형 진단
max_count = max(class_counts.values())
min_count = min(class_counts.values())
imbalance_ratio = max_count / min_count
print(f"📊 불균형 분석:")
print(f"   최대 클래스: {max_count}개")
print(f"   최소 클래스: {min_count}개")
print(f"   불균형 비율: {imbalance_ratio:.1f}x", end="")
print(" (⚠️ 불균형 심함, WeightedSampler 필요)" if imbalance_ratio > 3 else " (✅ 양호)")

### CELL 4: 클래스별 샘플 이미지 시각화

**학습 포인트:**
- 모델 학습 전 데이터를 눈으로 확인하는 것은 필수!
- `PIL.Image.open().convert("RGB")` : RGBA, Grayscale 등 다양한 포맷을 RGB로 통일
- `plt.subplots(2, 5)` : 2행 5열 = 10개 클래스 동시 표시
- 이미지 크기/품질/라벨 오류 등을 사전에 발견 가능

In [ ]:
# ============================================================
# CELL 4: 클래스별 샘플 이미지 시각화
# ============================================================
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 5, figsize=(18, 8))
fig.suptitle("📸 10개 클래스 샘플 이미지", fontsize=16, fontweight='bold')
axes = axes.flatten()

for i, cls in enumerate(class_names):
    if cls in sample_images:
        img = Image.open(sample_images[cls]).convert("RGB")
        axes[i].imshow(img)
        axes[i].set_title(f"{cls}\n({class_counts[cls]} imgs)",
                          fontsize=8, fontweight='bold')
    axes[i].axis('off')  # 축 숨기기

plt.tight_layout()
sample_path = os.path.join(OUTPUT_PATH, "00_sample_images.png")
plt.savefig(sample_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"💾 샘플 이미지 저장: {sample_path}")

### CELL 5: 클래스 분포 막대 차트

**학습 포인트:**
- 클래스 불균형 시각화 → 어느 클래스 데이터가 적은지 파악
- `plt.cm.Set3.colors` : matplotlib 내장 색상 팔레트 (10색)
- 막대 위 숫자 표시 → 정확한 수치 확인

In [ ]:
# ============================================================
# CELL 5: 클래스 분포 막대 차트
# ============================================================
fig, ax = plt.subplots(figsize=(13, 5))

# Set3 색상 팔레트로 막대 그리기
colors = plt.cm.Set3.colors[:len(class_names)]
bars = ax.bar(
    range(len(class_names)),
    [class_counts[c] for c in class_names],
    color=colors,
    edgecolor='gray',
    linewidth=0.5
)

# 축 레이블
ax.set_xticks(range(len(class_names)))
ax.set_xticklabels(class_names, rotation=30, ha='right', fontsize=9)
ax.set_ylabel("Image Count", fontsize=12)
ax.set_title("📊 클래스별 이미지 수 분포", fontweight='bold', fontsize=13)
ax.grid(axis='y', alpha=0.3)

# 막대 위에 수치 표시
for bar, cls in zip(bars, class_names):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        str(class_counts[cls]),
        ha='center', va='bottom', fontsize=8, fontweight='bold'
    )

# 평균선 추가
avg = total_images / len(class_names)
ax.axhline(y=avg, color='red', linestyle='--', alpha=0.6, label=f'평균: {avg:.0f}개')
ax.legend()

plt.tight_layout()
dist_path = os.path.join(OUTPUT_PATH, "01_class_distribution.png")
plt.savefig(dist_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"💾 분포 차트 저장: {dist_path}")

---
## ⚙️ STEP 3. 데이터 전처리 (Data Preprocessing)

### CELL 6: 하이퍼파라미터 설정

**학습 포인트:**
- `CFG` 딕셔너리로 모든 하이퍼파라미터를 한 곳에서 관리 → 실험 관리 용이
- `img_size=300` : EfficientNet-B3의 권장 입력 해상도
- `batch_size=32` : GPU 메모리와 학습 안정성의 균형
- `patience=7` : 7 에포크 동안 성능이 향상되지 않으면 학습 조기 종료
- `label_smoothing=0.1` : 과적합 방지 (정답 레이블을 0.9, 나머지에 0.1 분배)

| 파라미터 | 값 | 의미 |
|----------|-----|------|
| img_size | 300 | EfficientNet-B3 최적 해상도 |
| batch_size | 32 | 한 번에 처리할 이미지 수 |
| num_epochs | 30 | 최대 학습 반복 횟수 |
| lr | 1e-4 | AdamW 초기 학습률 |
| patience | 7 | Early Stopping 기준 |
| train_ratio | 0.8 | 학습/검증/테스트 = 80:10:10 |

In [ ]:
# ============================================================
# CELL 6: 하이퍼파라미터 설정
# ============================================================
import torch
import numpy as np

CFG = {
    # ── 모델 설정 ──────────────────────────────
    "model_name"  : "efficientnet_b3",  # timm 모델명
    "img_size"    : 300,                # EfficientNet-B3 권장 해상도
    "num_classes" : len(class_names),   # 자동으로 클래스 수 설정

    # ── 학습 설정 ──────────────────────────────
    "batch_size"  : 32,
    "num_epochs"  : 30,
    "lr"          : 1e-4,               # AdamW 초기 학습률
    "weight_decay": 1e-4,              # L2 정규화 계수
    "patience"    : 7,                  # Early Stopping 대기 에포크

    # ── 데이터 분할 ────────────────────────────
    "train_ratio" : 0.8,               # 80% 학습
    "val_ratio"   : 0.1,               # 10% 검증
    "test_ratio"  : 0.1,               # 10% 테스트

    # ── 기타 ────────────────────────────────────
    "num_workers" : 2,                  # 데이터 로딩 병렬 프로세스 수
    "seed"        : 42,                 # 재현성을 위한 시드
}

# ── GPU/CPU 자동 선택 ──────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⚡ Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"   GPU 모델: {torch.cuda.get_device_name(0)}")
    print(f"   GPU 메모리: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("   ⚠️ GPU 없음 - CPU로 실행 (매우 느릴 수 있음)")

# ── 재현성(Reproducibility) 시드 고정 ─────────────────────
torch.manual_seed(CFG["seed"])
np.random.seed(CFG["seed"])

print(f"\n📋 하이퍼파라미터:")
for k, v in CFG.items():
    print(f"   {k:20s}: {v}")

### CELL 7: 데이터 증강 (Data Augmentation)

**학습 포인트:**
- **데이터 증강** : 기존 이미지를 변환하여 데이터 다양성 확보 → 과적합 방지
- 학습 데이터(`train_transform`)에만 증강 적용, 검증/테스트(`val_transform`)는 변환만
- `Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])` : ImageNet 통계값 사용 (전이학습 필수)

| 변환 | 적용 대상 | 효과 |
|------|-----------|------|
| RandomHorizontalFlip | Train | 좌우 반전으로 위치 불변성 |
| RandomVerticalFlip | Train | 상하 반전 |
| RandomRotation(15) | Train | ±15° 회전 |
| ColorJitter | Train | 밝기/대비/채도 변화 |
| RandomAffine | Train | 이동 변환 |
| Normalize | Train+Val | ImageNet 정규화 |

In [ ]:
# ============================================================
# CELL 7: 데이터 증강 (Data Augmentation)
# ============================================================
from torchvision import transforms

# ── 학습용 변환 (증강 포함) ────────────────────────────────
train_transform = transforms.Compose([
    # 1. 크기 통일 (모델 입력 크기)
    transforms.Resize((CFG["img_size"], CFG["img_size"])),

    # 2. 기하학적 증강 (위치/방향 불변성)
    transforms.RandomHorizontalFlip(),                      # 50% 확률 좌우 반전
    transforms.RandomVerticalFlip(),                        # 50% 확률 상하 반전
    transforms.RandomRotation(15),                          # ±15° 회전
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),  # ±5% 이동

    # 3. 색상 증강 (조명 변화에 강건성)
    transforms.ColorJitter(
        brightness=0.2,   # 밝기 ±20%
        contrast=0.2,     # 대비 ±20%
        saturation=0.1    # 채도 ±10%
    ),

    # 4. Tensor 변환 + 정규화
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],  # ImageNet RGB 평균
        std=[0.229, 0.224, 0.225]    # ImageNet RGB 표준편차
    ),
])

# ── 검증/테스트용 변환 (증강 없음, 크기+정규화만) ──────────
val_transform = transforms.Compose([
    transforms.Resize((CFG["img_size"], CFG["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

print("✅ 데이터 변환 파이프라인 정의 완료")
print(f"\n🔄 Train Transform:")
print(train_transform)
print(f"\n🔄 Val/Test Transform:")
print(val_transform)

### CELL 8: 데이터셋 분할 및 DataLoader 생성

**학습 포인트:**
- `ImageFolder` : 폴더 구조를 자동으로 클래스로 인식하는 torchvision 표준 데이터셋
- `MultiExtImageFolder` : 다양한 이미지 확장자 지원을 위한 커스텀 클래스
- `Subset` : 전체 데이터셋에서 특정 인덱스만 선택
- `WeightedRandomSampler` : 클래스 불균형 해소 → 적은 클래스는 더 자주 샘플링
- `pin_memory=True` : CPU→GPU 데이터 전송 속도 향상 (GPU 학습 시)

In [ ]:
# ============================================================
# CELL 8: 데이터셋 분할 및 DataLoader 생성
# ============================================================
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, WeightedRandomSampler, Subset

# ── 커스텀 Dataset: 다양한 이미지 확장자 지원 ─────────────
class MultiExtImageFolder(ImageFolder):
    """png/jpg/jpeg/bmp/tiff 등 모든 이미지 포맷을 지원하는 ImageFolder"""
    def is_valid_file(self, path):
        return pathlib.Path(path).suffix.lower() in IMG_EXTS

# ── 전체 데이터셋 로드 (인덱스 분리용, transform 없음) ────
full_dataset = MultiExtImageFolder(root=INPUT_PATH)
n = len(full_dataset)

# ── 학습/검증/테스트 개수 계산 ────────────────────────────
n_train = int(n * CFG["train_ratio"])         # 80%
n_val   = int(n * CFG["val_ratio"])           # 10%
n_test  = n - n_train - n_val                 # 나머지 10%

# ── 무작위 인덱스 분리 ─────────────────────────────────────
indices = list(range(n))
np.random.shuffle(indices)                    # 시드 고정으로 재현성 보장
train_idx = indices[:n_train]
val_idx   = indices[n_train:n_train + n_val]
test_idx  = indices[n_train + n_val:]

# ── 각 분할에 적절한 transform 적용하여 Dataset 생성 ─────
train_ds = Subset(MultiExtImageFolder(root=INPUT_PATH, transform=train_transform), train_idx)
val_ds   = Subset(MultiExtImageFolder(root=INPUT_PATH, transform=val_transform),   val_idx)
test_ds  = Subset(MultiExtImageFolder(root=INPUT_PATH, transform=val_transform),   test_idx)

print(f"📦 데이터셋 분할 결과:")
print(f"   전체   : {n:6d}개")
print(f"   Train  : {len(train_ds):6d}개 ({CFG['train_ratio']*100:.0f}%)")
print(f"   Val    : {len(val_ds):6d}개 ({CFG['val_ratio']*100:.0f}%)")
print(f"   Test   : {len(test_ds):6d}개 ({CFG['test_ratio']*100:.0f}%)")

# ── WeightedRandomSampler: 클래스 불균형 해소 ─────────────
# 학습 데이터의 클래스 레이블 추출
labels_train = [full_dataset.targets[i] for i in train_idx]
# 클래스별 샘플 수 계산
class_sample_count = np.array([labels_train.count(i) for i in range(CFG["num_classes"])])
# 클래스별 가중치 계산 (샘플 수 역수 → 적은 클래스일수록 높은 가중치)
weights = 1.0 / np.maximum(class_sample_count, 1)
# 각 샘플에 클래스 가중치 할당
sample_weights = np.array([weights[l] for l in labels_train])
sampler = WeightedRandomSampler(
    torch.from_numpy(sample_weights).float(),
    num_samples=len(sample_weights),
    replacement=True       # 복원 추출 (불균형 클래스 반복 사용)
)

# ── DataLoader 생성 ────────────────────────────────────────
train_loader = DataLoader(
    train_ds,
    batch_size=CFG["batch_size"],
    sampler=sampler,              # WeightedSampler 적용 (shuffle 대신)
    num_workers=CFG["num_workers"],
    pin_memory=(DEVICE == "cuda") # GPU 사용 시 메모리 전송 최적화
)
val_loader = DataLoader(
    val_ds,
    batch_size=CFG["batch_size"],
    shuffle=False,                # 검증은 순서 유지
    num_workers=CFG["num_workers"]
)
test_loader = DataLoader(
    test_ds,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=CFG["num_workers"]
)

print(f"\n✅ DataLoader 생성 완료")
print(f"   배치 크기: {CFG['batch_size']}")
print(f"   Train 배치 수: {len(train_loader)}")
print(f"   Val   배치 수: {len(val_loader)}")
print(f"   Test  배치 수: {len(test_loader)}")

---
## 🧠 STEP 4. 모델 정의 (Model Definition)

### CELL 9: EfficientNet-B3 모델 로드 (전이학습)

**학습 포인트:**
- **전이학습(Transfer Learning)** : ImageNet으로 사전 학습된 가중치를 의료 이미지에 적용
- `timm.create_model(pretrained=True)` : ImageNet 사전학습 가중치 자동 다운로드
- `num_classes=10` : 마지막 분류 층을 10개 클래스로 교체
- EfficientNet-B3은 B0 대비 높은 정확도, B7 대비 빠른 속도의 균형점

**EfficientNet 스케일링 비교:**

| 모델 | 입력 크기 | 파라미터 | Top-1 Acc |
|------|-----------|----------|------------|
| B0 | 224 | 5.3M | 77.1% |
| **B3** | **300** | **12M** | **81.1%** |
| B7 | 600 | 66M | 84.3% |

In [ ]:
# ============================================================
# CELL 9: EfficientNet-B3 모델 로드 (전이학습)
# ============================================================
import timm
import torch.nn as nn

# ── 사전학습 모델 생성 ─────────────────────────────────────
# pretrained=True : ImageNet-1K 사전학습 가중치 사용
# num_classes=10  : 최종 분류층을 10개 출력으로 교체
model = timm.create_model(
    CFG["model_name"],
    pretrained=True,
    num_classes=CFG["num_classes"]
)
model = model.to(DEVICE)

# ── 파라미터 수 확인 ────────────────────────────────────────
total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"🧠 모델: {CFG['model_name']}")
print(f"   전체 파라미터  : {total_params:,}")
print(f"   학습 가능 파라미터: {trainable_params:,}")
print(f"   모델 크기 (추정): {total_params * 4 / 1e6:.1f} MB")
print(f"   디바이스: {DEVICE}")

# ── 입력/출력 형태 확인 ────────────────────────────────────
print(f"\n📐 입력/출력 크기 확인:")
dummy = torch.randn(1, 3, CFG["img_size"], CFG["img_size"]).to(DEVICE)
with torch.no_grad():
    out = model(dummy)
print(f"   입력: {dummy.shape}  →  출력: {out.shape}")
print(f"   (배치크기, 채널, 높이, 너비) → (배치크기, 클래스수)")

### CELL 10: 손실함수 및 옵티마이저 설정

**학습 포인트:**
- **CrossEntropyLoss** : 다중 클래스 분류의 표준 손실함수
  - `weight` : 클래스별 가중치 → 불균형 보정 (희귀 클래스 오분류에 더 큰 패널티)
  - `label_smoothing=0.1` : 정답 레이블을 0.9로 완화 → 과신뢰(overconfidence) 방지
- **AdamW** : Adam + Weight Decay 수정 → 현대 딥러닝의 표준 옵티마이저
- **CosineAnnealingLR** : 학습률을 코사인 곡선으로 서서히 감소 → 안정적 수렴

In [ ]:
# ============================================================
# CELL 10: 손실함수, 옵티마이저, 스케줄러 설정
# ============================================================

# ── 클래스 불균형 반영한 가중치 계산 ─────────────────────
# 가중치 = 1/클래스별샘플수 (정규화하여 합이 num_classes가 되도록)
class_weights_tensor = torch.tensor(
    weights / weights.sum() * CFG["num_classes"],
    dtype=torch.float32
).to(DEVICE)

# ── 손실함수: CrossEntropyLoss ────────────────────────────
criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor,  # 클래스 불균형 보정
    label_smoothing=0.1            # 과적합 방지 (0.0 ~ 0.2 사이 권장)
)

# ── 옵티마이저: AdamW ─────────────────────────────────────
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG["lr"],                  # 초기 학습률
    weight_decay=CFG["weight_decay"]  # L2 정규화
)

# ── 학습률 스케줄러: CosineAnnealingLR ────────────────────
# 학습률을 코사인 곡선으로 감소: lr → eta_min
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CFG["num_epochs"],       # 주기 (전체 에포크)
    eta_min=1e-6                   # 최소 학습률
)

print("✅ 학습 구성 요소 설정 완료")
print(f"   손실함수   : CrossEntropyLoss (weight + label_smoothing=0.1)")
print(f"   옵티마이저 : AdamW (lr={CFG['lr']}, wd={CFG['weight_decay']})")
print(f"   스케줄러   : CosineAnnealingLR (T_max={CFG['num_epochs']}, eta_min=1e-6)")

print(f"\n📊 클래스별 손실 가중치:")
for cls, w in zip(class_names, class_weights_tensor.cpu()):
    print(f"   {cls:35s}: {w:.4f}")

### CELL 11: 학습 보조 함수 정의

**학습 포인트:**
- 1 에포크 학습과 검증을 각각 함수로 분리 → 코드 가독성 향상
- `model.train()` / `model.eval()` : BatchNorm, Dropout 등의 동작 모드 전환
- `torch.no_grad()` : 검증 시 그래디언트 계산 비활성화 → 메모리/속도 효율
- `clip_grad_norm_` : 그래디언트 폭발(gradient explosion) 방지

In [ ]:
# ============================================================
# CELL 11: 학습/검증 보조 함수 정의
# ============================================================
from tqdm.notebook import tqdm

def train_one_epoch(model, loader, criterion, optimizer, device):
    """1 에포크 학습 함수
    Returns: (평균 손실, 정확도%)
    """
    model.train()  # 학습 모드: Dropout, BatchNorm 활성화
    total_loss, correct, total = 0.0, 0, 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)

        # 1. 그래디언트 초기화
        optimizer.zero_grad()

        # 2. 순전파 (Forward pass)
        outputs = model(imgs)

        # 3. 손실 계산
        loss = criterion(outputs, labels)

        # 4. 역전파 (Backward pass)
        loss.backward()

        # 5. 그래디언트 클리핑 (폭발 방지)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # 6. 가중치 업데이트
        optimizer.step()

        # 통계 집계
        total_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)

    return total_loss / total, correct / total * 100


def evaluate(model, loader, criterion, device):
    """검증/테스트 평가 함수
    Returns: (평균 손실, 정확도%)
    """
    model.eval()   # 평가 모드: Dropout 비활성화, BatchNorm 고정
    total_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():  # 그래디언트 계산 비활성화 (메모리/속도 절약)
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss    = criterion(outputs, labels)

            total_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += imgs.size(0)

    return total_loss / total, correct / total * 100


print("✅ 학습/검증 함수 정의 완료")
print("   - train_one_epoch(): 1 에포크 학습")
print("   - evaluate(): 손실 및 정확도 평가")

---
## 🚀 STEP 5. 모델 학습 (Model Training)

### CELL 12: 학습 루프 실행

**학습 포인트:**
- 매 에포크마다 학습 → 검증 → 결과 기록 → 최적 모델 저장
- **Early Stopping** : 검증 정확도가 `patience` 에포크 동안 향상 없으면 조기 종료
- `torch.save(checkpoint)` : 에포크, 가중치, 설정 등을 묶어 저장 (재개 가능)
- 체크포인트에 `class_names`, `cfg` 포함 → 나중에 불러올 때 추가 정보 불필요

> ⏱️ EfficientNet-B3 + 300×300 + GPU 기준 에포크당 약 3~8분 소요

In [ ]:
# ============================================================
# CELL 12: 학습 루프 실행
# ============================================================

# ── 기록용 딕셔너리 초기화 ────────────────────────────────
history = {
    "train_loss": [], "val_loss": [],
    "train_acc":  [], "val_acc":  []
}

best_val_acc   = 0.0   # 최고 검증 정확도
patience_count = 0     # Early Stopping 카운터
best_ckpt      = None  # 최적 체크포인트 경로

print("🚀 학습 시작!")
print("=" * 70)

for epoch in range(1, CFG["num_epochs"] + 1):

    # ── 1. 학습 ───────────────────────────────────────────
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, DEVICE
    )

    # ── 2. 검증 ───────────────────────────────────────────
    val_loss, val_acc = evaluate(
        model, val_loader, criterion, DEVICE
    )

    # ── 3. 학습률 스케줄러 업데이트 ───────────────────────
    scheduler.step()
    lr_now = optimizer.param_groups[0]["lr"]

    # ── 4. 기록 저장 ──────────────────────────────────────
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    # ── 5. 최적 모델 저장 (Best Val Acc 갱신 시) ──────────
    flag = ""
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_count = 0
        best_ckpt = os.path.join(OUTPUT_PATH, "best_model.pth")
        torch.save({
            "epoch"               : epoch,
            "model_state_dict"    : model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_acc"             : val_acc,
            "class_names"         : class_names,
            "cfg"                 : CFG,
        }, best_ckpt)
        flag = "  ⭐ Best!"
    else:
        patience_count += 1

    # ── 6. 로그 출력 ──────────────────────────────────────
    print(
        f"Epoch {epoch:02d}/{CFG['num_epochs']} | "
        f"LR: {lr_now:.2e} | "
        f"Train Loss: {train_loss:.4f}  Acc: {train_acc:.2f}% | "
        f"Val Loss: {val_loss:.4f}  Acc: {val_acc:.2f}%"
        f"{flag}"
    )

    # ── 7. Early Stopping 판단 ────────────────────────────
    if patience_count >= CFG["patience"]:
        print(f"\n⏹️  Early Stopping! (patience={CFG['patience']} 에포크 동안 개선 없음)")
        break

print(f"\n{'='*70}")
print(f"✅ 학습 완료  |  Best Val Acc: {best_val_acc:.2f}%")
print(f"   체크포인트: {best_ckpt}")

### CELL 13: 학습 곡선 시각화 및 저장

**학습 포인트:**
- **Loss 곡선** : Train Loss ↓, Val Loss ↓ → 정상 학습
- **과적합 진단**: Train Acc ↑, Val Acc ↓ → 과적합(Overfitting) → 정규화 강화 필요
- **미학습 진단**: 두 곡선 모두 낮음 → 모델 용량 부족 or 학습률 문제

In [ ]:
# ============================================================
# CELL 13: 학습 곡선 시각화 및 저장
# ============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ep = range(1, len(history["train_loss"]) + 1)

# ── Loss 곡선 ─────────────────────────────────────────────
ax1.plot(ep, history["train_loss"], label="Train",
         color="#2196F3", marker='o', markersize=3, linewidth=1.5)
ax1.plot(ep, history["val_loss"],   label="Val",
         color="#F44336", marker='o', markersize=3, linewidth=1.5)
ax1.set_title("📉 Loss Curve", fontweight='bold', fontsize=13)
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid(alpha=0.3)

# ── Accuracy 곡선 ─────────────────────────────────────────
ax2.plot(ep, history["train_acc"], label="Train",
         color="#2196F3", marker='o', markersize=3, linewidth=1.5)
ax2.plot(ep, history["val_acc"],   label="Val",
         color="#F44336", marker='o', markersize=3, linewidth=1.5)
ax2.set_title("📈 Accuracy Curve", fontweight='bold', fontsize=13)
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy (%)")
ax2.legend()
ax2.grid(alpha=0.3)
ax2.axhline(y=80, color='gray', linestyle='--', alpha=0.4, label='80%')

# 최적 에포크 표시
best_epoch = history["val_acc"].index(max(history["val_acc"])) + 1
ax2.axvline(x=best_epoch, color='green', linestyle=':', alpha=0.7,
            label=f'Best Epoch ({best_epoch})')
ax2.legend()

plt.tight_layout()
curve_path = os.path.join(OUTPUT_PATH, "02_training_curve.png")
plt.savefig(curve_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"💾 학습 곡선 저장: {curve_path}")
print(f"   최고 Val Acc: {best_val_acc:.2f}% (Epoch {best_epoch})")

---
## 📊 STEP 6. 모델 평가 (Model Evaluation)

### CELL 14: 최적 모델 로드 및 테스트 추론

**학습 포인트:**
- `torch.load(best_ckpt)` : 저장된 체크포인트 복원
- `model.load_state_dict()` : 체크포인트의 가중치를 현재 모델에 적용
- `torch.softmax()` : 로짓(logit)을 확률로 변환 → 각 클래스의 예측 확률 해석 가능
- 테스트 세트는 **학습/검증에서 전혀 사용하지 않은** 데이터 → 실제 성능의 공정한 측정

In [ ]:
# ============================================================
# CELL 14: 최적 모델 로드 및 테스트 추론
# ============================================================
import numpy as np

# ── 최적 체크포인트 로드 ───────────────────────────────────
print(f"📂 Best 모델 로드: {best_ckpt}")
checkpoint = torch.load(best_ckpt, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
print(f"   저장된 Val Acc: {checkpoint['val_acc']:.2f}% (Epoch {checkpoint['epoch']})")

# ── 테스트 데이터 추론 ─────────────────────────────────────
model.eval()
all_preds, all_labels, all_probs = [], [], []

print("\n🔍 테스트 데이터 추론 중...")
with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc="Testing"):
        imgs = imgs.to(DEVICE)
        outputs = model(imgs)

        # 확률 계산 (Softmax)
        probs = torch.softmax(outputs, dim=1)
        preds = outputs.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

# numpy 배열로 변환
all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)

# ── 전체 정확도 계산 ───────────────────────────────────────
test_acc = (all_preds == all_labels).mean() * 100
print(f"\n🎯 테스트 정확도 (Test Accuracy): {test_acc:.2f}%")
print(f"   테스트 샘플 수: {len(all_labels)}개")
print(f"   정확히 예측된 수: {(all_preds == all_labels).sum()}개")

### CELL 15: Confusion Matrix 시각화

**학습 포인트:**
- **Confusion Matrix** : 실제 클래스(행) vs 예측 클래스(열) → 어떤 클래스가 혼동되는지 파악
- 대각선 = 정분류, 비대각선 = 오분류
- `seaborn.heatmap` : 색상 강도로 혼동 정도 직관적 표시
- 의료 영상 분류에서는 특히 **False Negative** (골절을 정상으로 분류)를 최소화하는 것이 중요

In [ ]:
# ============================================================
# CELL 15: Confusion Matrix 시각화 및 저장
# ============================================================
from sklearn.metrics import confusion_matrix
import seaborn as sns

# ── Confusion Matrix 계산 ─────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)

# 클래스명 단축 (시각화 공간 절약)
short_names = [
    c.replace(" fracture", "").replace(" Fracture", "")
    for c in class_names
]

# ── 시각화 ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(
    cm,
    annot=True,           # 셀 안에 숫자 표시
    fmt='d',              # 정수 형식
    cmap='Blues',         # 파란색 계열 색상
    xticklabels=short_names,
    yticklabels=short_names,
    ax=ax,
    linewidths=0.5,       # 셀 경계선
    linecolor='gray'
)
ax.set_xlabel("예측 클래스 (Predicted)", fontsize=12, fontweight='bold')
ax.set_ylabel("실제 클래스 (True)",      fontsize=12, fontweight='bold')
ax.set_title(
    f"Confusion Matrix  (Test Acc: {test_acc:.2f}%)",
    fontsize=14, fontweight='bold'
)
plt.xticks(rotation=30, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()

cm_path = os.path.join(OUTPUT_PATH, "03_confusion_matrix.png")
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"💾 Confusion Matrix 저장: {cm_path}")

### CELL 16: Classification Report (정밀도/재현율/F1)

**학습 포인트:**
- **Precision (정밀도)** : 예측 양성 중 실제 양성 비율 → 오탐(False Positive) 최소화
- **Recall (재현율)** : 실제 양성 중 예측 양성 비율 → 미탐(False Negative) 최소화
- **F1-Score** : Precision과 Recall의 조화평균 → 불균형 데이터에서 종합 지표
- **Support** : 각 클래스의 실제 테스트 샘플 수

```
의료 영상 분류에서:
  Recall (재현율) 우선 → 골절을 놓치는 것이 더 위험
```

In [ ]:
# ============================================================
# CELL 16: Classification Report 출력 및 저장
# ============================================================
from sklearn.metrics import classification_report

# ── 전체 리포트 생성 ──────────────────────────────────────
report = classification_report(
    all_labels,
    all_preds,
    target_names=class_names,
    digits=4               # 소수점 4자리 출력
)

print("📋 Classification Report")
print("=" * 80)
print(report)

# ── 텍스트 파일로 저장 ────────────────────────────────────
report_path = os.path.join(OUTPUT_PATH, "04_classification_report.txt")
with open(report_path, "w", encoding="utf-8") as f:
    f.write(f"=" * 60 + "\n")
    f.write(f"Bone Fracture 10-Class Classification Report\n")
    f.write(f"Model: {CFG['model_name']}\n")
    f.write(f"=" * 60 + "\n\n")
    f.write(f"Test  Accuracy : {test_acc:.4f}%\n")
    f.write(f"Best Val Acc   : {best_val_acc:.4f}%\n\n")
    f.write(report)

print(f"💾 Classification Report 저장: {report_path}")

---
## 📈 STEP 7. 결과 시각화 (Result Visualization)

### CELL 17: 클래스별 정확도 막대 차트

**학습 포인트:**
- Confusion Matrix 대각선 / 각 행의 합 = 클래스별 정확도(Recall)
- 색상 코딩 : 🟢 80% 이상 / 🟠 60~80% / 🔴 60% 미만
- 어떤 골절 유형이 가장 분류하기 어려운지 파악 → 추가 데이터 수집 대상 식별

In [ ]:
# ============================================================
# CELL 17: 클래스별 정확도 (Per-Class Accuracy) 바 차트
# ============================================================

# ── 클래스별 정확도 계산 (Confusion Matrix 대각선 활용) ────
per_class_acc = cm.diagonal() / cm.sum(axis=1) * 100

# ── 성능에 따른 색상 코딩 ────────────────────────────────
colors = [
    '#4CAF50' if a >= 80    # 녹색: 우수 (80% 이상)
    else '#FF9800' if a >= 60  # 주황: 보통 (60~80%)
    else '#F44336'             # 빨강: 미흡 (60% 미만)
    for a in per_class_acc
]

fig, ax = plt.subplots(figsize=(13, 6))
bars = ax.bar(range(len(class_names)), per_class_acc, color=colors, edgecolor='white')

ax.set_xticks(range(len(class_names)))
ax.set_xticklabels(class_names, rotation=30, ha='right', fontsize=9)
ax.set_ylabel("정확도 (Accuracy, %)", fontsize=12)
ax.set_title("클래스별 테스트 정확도 (Per-Class Test Accuracy)",
             fontweight='bold', fontsize=13)
ax.set_ylim(0, 115)

# 기준선
ax.axhline(y=80, color='green', linestyle='--', alpha=0.5, label='80% 기준선')
ax.axhline(y=60, color='orange', linestyle='--', alpha=0.5, label='60% 기준선')

# 막대 위 수치 표시
for bar, acc in zip(bars, per_class_acc):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f"{acc:.1f}%",
        ha='center', va='bottom', fontsize=8, fontweight='bold'
    )

# 범례 (색상 의미)
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#4CAF50', label='우수 (≥80%)'),
    Patch(facecolor='#FF9800', label='보통 (60~80%)'),
    Patch(facecolor='#F44336', label='미흡 (<60%)'),
]
ax.legend(handles=legend_elements, loc='upper right')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
peracc_path = os.path.join(OUTPUT_PATH, "05_per_class_accuracy.png")
plt.savefig(peracc_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"💾 Per-Class Accuracy 저장: {peracc_path}")
print(f"\n📊 클래스별 정확도 요약:")
for cls, acc in zip(class_names, per_class_acc):
    status = "✅" if acc >= 80 else "⚠️" if acc >= 60 else "❌"
    print(f"   {status} {cls:35s}: {acc:.1f}%")

### CELL 18: 예측 신뢰도 분석 (Top-K 확률 시각화)

**학습 포인트:**
- 모델이 예측할 때 얼마나 확신(confidence)하는지 분석
- **높은 신뢰도 + 오분류** → 모델이 잘못된 확신을 갖는 위험한 상황
- **낮은 신뢰도 + 정분류** → 모델이 불확실하게 맞춘 경우 → 추가 학습 필요

In [ ]:
# ============================================================
# CELL 18: 예측 신뢰도 분포 시각화
# ============================================================

# ── 최대 예측 확률 (예측 신뢰도) 추출 ────────────────────
max_probs = all_probs.max(axis=1)  # 각 샘플의 최대 예측 확률
correct_mask   = (all_preds == all_labels)  # 정분류 여부
incorrect_mask = ~correct_mask

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── 왼쪽: 전체 신뢰도 히스토그램 ─────────────────────────
axes[0].hist(max_probs[correct_mask],   bins=20, alpha=0.7,
             label=f'정분류 (n={correct_mask.sum()})', color='#4CAF50')
axes[0].hist(max_probs[incorrect_mask], bins=20, alpha=0.7,
             label=f'오분류 (n={incorrect_mask.sum()})', color='#F44336')
axes[0].set_xlabel("예측 확률 (신뢰도)"); axes[0].set_ylabel("샘플 수")
axes[0].set_title("예측 신뢰도 분포", fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)

# ── 오른쪽: 클래스별 평균 신뢰도 ─────────────────────────
class_avg_conf = [
    max_probs[all_labels == i].mean()
    for i in range(CFG["num_classes"])
]
axes[1].bar(range(len(class_names)), class_avg_conf,
            color=plt.cm.Paired.colors[:len(class_names)])
axes[1].set_xticks(range(len(class_names)))
axes[1].set_xticklabels(class_names, rotation=30, ha='right', fontsize=8)
axes[1].set_ylabel("평균 예측 신뢰도")
axes[1].set_title("클래스별 평균 예측 신뢰도", fontweight='bold')
axes[1].set_ylim(0, 1.1); axes[1].grid(axis='y', alpha=0.3)

for i, conf in enumerate(class_avg_conf):
    axes[1].text(i, conf + 0.01, f"{conf:.2f}",
                 ha='center', va='bottom', fontsize=7)

plt.tight_layout()
conf_path = os.path.join(OUTPUT_PATH, "06_confidence_analysis.png")
plt.savefig(conf_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"💾 신뢰도 분석 저장: {conf_path}")
print(f"\n   전체 평균 신뢰도: {max_probs.mean():.3f}")
print(f"   정분류 평균 신뢰도: {max_probs[correct_mask].mean():.3f}")
print(f"   오분류 평균 신뢰도: {max_probs[incorrect_mask].mean():.3f}")

---
## 💾 STEP 8. 결과 저장 (Save Results)

### CELL 19: 실험 결과 요약 JSON 저장

**학습 포인트:**
- 모든 실험 결과를 JSON 파일로 저장 → 실험 재현/비교 용이
- `ensure_ascii=False` : 한글 문자 깨짐 방지
- `indent=2` : JSON 들여쓰기로 가독성 향상
- 타임스탬프 + 하이퍼파라미터 + 결과를 함께 저장 → 실험 추적(Experiment Tracking)

In [ ]:
# ============================================================
# CELL 19: 실험 결과 요약 JSON 저장
# ============================================================
import json

summary = {
    # ── 실험 메타데이터 ──────────────────────────────────
    "run_timestamp"    : RUN_STAMP,
    "model"            : CFG["model_name"],
    "num_classes"      : CFG["num_classes"],
    "class_names"      : class_names,
    "input_path"       : INPUT_PATH,
    "output_path"      : OUTPUT_PATH,

    # ── 데이터셋 정보 ─────────────────────────────────────
    "dataset_total"    : total_images,
    "train_samples"    : len(train_ds),
    "val_samples"      : len(val_ds),
    "test_samples"     : len(test_ds),

    # ── 학습 결과 ─────────────────────────────────────────
    "epochs_trained"   : len(history["train_loss"]),
    "best_val_acc_pct" : round(best_val_acc, 4),
    "test_acc_pct"     : round(test_acc, 4),

    # ── 하이퍼파라미터 ────────────────────────────────────
    "hyperparams"      : CFG,

    # ── 클래스별 정확도 ───────────────────────────────────
    "per_class_acc"    : {
        cls: round(float(acc), 2)
        for cls, acc in zip(class_names, per_class_acc)
    },

    # ── 학습 곡선 데이터 ──────────────────────────────────
    "history"          : {
        "train_loss": [round(v, 6) for v in history["train_loss"]],
        "val_loss"  : [round(v, 6) for v in history["val_loss"]],
        "train_acc" : [round(v, 4) for v in history["train_acc"]],
        "val_acc"   : [round(v, 4) for v in history["val_acc"]],
    }
}

summary_path = os.path.join(OUTPUT_PATH, "07_run_summary.json")
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"💾 실험 요약 저장: {summary_path}")

### CELL 20: 최종 결과 출력 및 저장 파일 목록 확인

**학습 포인트:**
- 전체 파이프라인 실행 결과를 최종 확인
- 생성된 파일 목록과 크기를 통해 저장 완료 여부 점검
- Best Val Acc와 Test Acc의 차이 → 일반화 성능 평가

In [ ]:
# ============================================================
# CELL 20: 최종 결과 출력 및 저장 파일 목록 확인
# ============================================================
import os

print(f"{'='*65}")
print(f"  🎉 Bone Fracture 10-Class Classification 완료!")
print(f"{'='*65}")
print(f"")
print(f"  📊 최종 성능 결과")
print(f"  {'─'*40}")
print(f"  ✅ Best Val Accuracy : {best_val_acc:.2f}%")
print(f"  ✅ Test Accuracy     : {test_acc:.2f}%")
print(f"  📅 학습 에포크 수     : {len(history['train_loss'])}")
print(f"  🧠 사용 모델         : {CFG['model_name']}")
print(f"")
print(f"  📁 저장된 파일 목록")
print(f"  {'─'*40}")

# 출력 폴더의 파일 목록 및 크기 출력
file_list = [
    "00_sample_images.png",
    "01_class_distribution.png",
    "02_training_curve.png",
    "03_confusion_matrix.png",
    "04_classification_report.txt",
    "05_per_class_accuracy.png",
    "06_confidence_analysis.png",
    "07_run_summary.json",
    "best_model.pth",
]

for fname in file_list:
    fpath = os.path.join(OUTPUT_PATH, fname)
    if os.path.exists(fpath):
        size = os.path.getsize(fpath)
        size_str = f"{size/1e6:.2f} MB" if size > 1e6 else f"{size/1e3:.1f} KB"
        print(f"  ✅ {fname:40s} ({size_str})")
    else:
        print(f"  ❌ {fname:40s} (미생성)")

print(f"")
print(f"  📂 출력 경로: {OUTPUT_PATH}")
print(f"{'='*65}")

print(f"\n📌 클래스별 최종 정확도 요약:")
sorted_results = sorted(
    zip(class_names, per_class_acc),
    key=lambda x: x[1], reverse=True
)
for rank, (cls, acc) in enumerate(sorted_results, 1):
    bar = '█' * int(acc // 5)
    print(f"  {rank:2d}. {cls:35s} {acc:5.1f}% {bar}")